# Figure S4b–c — program CRT and replicate effect sizes

This is a re-scripted workflow, not a notebook launcher. Source Q-Q and replicate-effect functions reside in `figure_s04_panels.py`; the original embedded CRT backend and replicate computation are ordinary repository Python modules.

The default reads numerical Q-Q/p-value and replicate beta results and redraws every panel. `RECOMPUTE_CRT=True` recalculates those data from the original inputs (B=10000; B_REP=127) using programDE. It does not run or require any old notebook, and never copies previously rendered figures.

Source: `iPSC_EC_QQ_full_covariates_with_replicate_correlation.ipynb`, original source SHA-256 `6e80625abd9b5e71d45da6dd19e888898318d39a159a7ac35600fc1d85489e87`. Parameters, NTC exclusions, pairing, correlation, and plotnine styling are ported.

In [ ]:
from pathlib import Path
import sys, os, json
from IPython.display import display
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "cc_perturbseq").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository checkout.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from cc_perturbseq.figure_notebook import initialize_figure, load_figure_manifest, get_path, panel_directory, write_receipt

import pandas as pd
from cc_perturbseq.figure_s04_panels import *
from cc_perturbseq import crt_qq_backend as qqb
from cc_perturbseq.figure_s04_runtime import build_crt_argv, launch_crt_recomputation
ctx = initialize_figure(load_figure_manifest("Figure_S04"), require_inputs=False)
PANEL_B_DIR, PANEL_C_DIR = panel_directory(ctx,"b"), panel_directory(ctx,"c")
# Replot from numerical results, never from old figure files. Enable raw CRT
# recomputation explicitly when the per-day results/beta matrices need rebuilding.
RECOMPUTE_CRT = False
# Set to "replicates" to resume a completed full-day export after a replicate failure.
# Its prior runtime/configuration/mapping records are verified before any output is changed.
CRT_STAGE = "all"
# Retain original gene/control group names and RNG inputs for manuscript reproduction.
GUIDE_MAPPING_POLICY = "source_names"
# Original S4 CRT used sc-dl; R-SCEPTRE guide assignment is a separate upstream step.
# Explicit override: /oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/bin/python
CRT_PYTHON = os.environ.get("CCPERTURBSEQ_S4_PYTHON", sys.executable)
print("Notebook Python:", sys.executable)
print("CRT worker Python:", CRT_PYTHON)
DAYS = ["d0","d1","d2","d3"]
K, DENSITY_THRESHOLD, B = 50, "0_2", 10000
NTC_LABELS = ["SAFE","non-targeting","NTC"]
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK","4"))
INPUT_MODEL_ROOT = Path(os.environ.get("CCPERTURBSEQ_S4_RESULTS", "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/iPSC_EC_QQ_full_covariates_with_replicate_correlation_bundle/results/ipsc_ec_qq_exports_full_covariates/k_50.dt_0_2"))
PROGRAMDE_ROOT = Path(qqb.DEFAULT_PROGRAMDE_ROOT)
DATA_DIR, CNMF_DIR = Path(qqb.DEFAULT_DATA_DIR), Path(qqb.DEFAULT_CNMF_DIR)
GUIDE_LONG_TEMPLATE = qqb.DEFAULT_GUIDE_LONG_TEMPLATE
GUIDE_METADATA_TSV = get_path(ctx["paths"],"ccperturbseq.target_metadata")
RECOMPUTED_ROOT = ctx["output_dir"] / "recomputed_crt"
print("Numerical inputs:",INPUT_MODEL_ROOT)
print("Recompute raw CRT:",RECOMPUTE_CRT)

# S4c settings from the FINAL executed source cell, not the earlier unfiltered panel.
# S4b uses empirical p-values; S4c's archived selection preferred skew p-values.
POOLED_PVALUE_TYPE = "skew"
POOLED_Q_CUTOFF = 0.10
MIN_TARGET_CELLS_PER_REP = 20
TARGET_COUNTS_CSV = None  # or an explicit DIAGNOSTIC_target_cell_counts_by_replicate.csv


## Numerical inputs and runtime

The archived program CRT/replicate notebook ran with
`/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/bin/python`.
The separate `ipsc_ec_sceptre.ipynb` used
`/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env/bin/python`
and `rpy2`/R to assign guides. This S4 notebook **reads those saved assignments**;
it does not run R-SCEPTRE or require switching its plotting kernel to R.

With `RECOMPUTE_CRT=True`, the code below validates the actual CLI arguments,
then starts the repository's Python CRT worker using `CRT_PYTHON`. The worker
checks its dependencies, programDE import location, and all requested day inputs
before any CRT. Both full-day CRT and replicate beta estimation run in that same
interpreter. No archived notebook is executed. The original calculation and
plotting functions remain in repository modules.

The default `CRT_PYTHON=sys.executable` follows the original notebook. Set it to
the explicit sc-dl Python path above to keep plotting in another kernel. A failed
worker stops the notebook; its traceback is retained in `recomputed_crt/s4_crt.log`.
The runtime/import/input report is `recomputed_crt/s4_crt_runtime.json`.


### Replicate-output flags and resuming

Replicate beta extraction uses `calibrate_skew_normal=False`,
`return_raw_pvals=False`, and `return_skew_normal=False`. `betas_df` is still
returned by programDE. The full-day skew-enabled call retains its original flags.
The optional no-skew path reads empirical results from `pvals_df`, not `pvals_raw_df`.

After a full-day export completed but replicate estimation failed, set
`RECOMPUTE_CRT=True` and `CRT_STAGE="replicates"`. The same output directory,
scientific settings, inputs, and guide mapping must match the completed run.
Missing/mismatched records stop before modifying outputs; use `CRT_STAGE="all"`
for a genuinely new or changed analysis. No completed p-values are silently
combined with a different guide mapping.

The v43 metadata schema (`guide_id`, `intended_target_name`) is now recognized.
Blank control targets retain their inferred control labels. The runtime report
records any target mapping differences from the previous name-based fallback.


`GUIDE_MAPPING_POLICY="source_names"` is the reproduction default. Metadata
labels are audited, but the existing gene/control grouping is preserved. The
checked-in v43 table contains aliases (for example T/TBXT) and individual SAFE
IDs in its target column. Those must NOT silently rename or split the original
CRT targets. `"metadata"` is an explicit alternative analysis mode; it retains
canonical control labels and requires regenerating both stages when mappings
change. The original completed export can be resumed under `"source_names"`.


In [ ]:
if RECOMPUTE_CRT:
    argv = build_crt_argv(
        programde_root=PROGRAMDE_ROOT, data_dir=DATA_DIR, cnmf_dir=CNMF_DIR,
        guide_long_template=GUIDE_LONG_TEMPLATE, guide_metadata_tsv=GUIDE_METADATA_TSV,
        output_dir=RECOMPUTED_ROOT, days=DAYS, k=K,
        density_threshold=DENSITY_THRESHOLD, B=B, n_jobs=N_JOBS,
        ntc_labels=NTC_LABELS, overwrite=True, guide_mapping_policy=GUIDE_MAPPING_POLICY,
    )
    # Uses the source parser's --guide-long-csv-template spelling. It is parsed
    # before a child process is started; Jupyter's own argv is never consumed.
    runtime_report = launch_crt_recomputation(argv, python_executable=CRT_PYTHON, stage=CRT_STAGE)
    display(runtime_report)
    INPUT_MODEL_ROOT = RECOMPUTED_ROOT / qqb.model_tag(K, qqb.normalize_density_token(DENSITY_THRESHOLD))

qq_csv = INPUT_MODEL_ROOT / "all_days_QQ_points.csv"
manifest_path = INPUT_MODEL_ROOT / "biological_replicate_effects/replicate_beta_manifest.csv"
for path in (qq_csv, manifest_path):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing numerical S4 input: {path}. Set INPUT_MODEL_ROOT to the results directory or enable RECOMPUTE_CRT. No previous notebook/figure is needed.")
qq_points = pd.read_csv(qq_csv)
# Recalculate Q-Q coordinates using the original backend rank formula.
if "pvalue" in qq_points:
    grouping = [c for c in ["day","pvalue_type","series_type"] if c in qq_points]
    qq_points = pd.concat([qqb.add_qq_columns(g.copy()) for _,g in qq_points.groupby(grouping,sort=False)],ignore_index=True)
qq_points.to_csv(PANEL_B_DIR / "all_days_QQ_points.csv",index=False)
manifest = pd.read_csv(manifest_path)
manifest = manifest.loc[manifest["status"].isin(["written","existing"]) & manifest["beta_path"].notna()].copy()
# Only explicit relative paths or the exact source basename in its day folder
# are relocated; never scan arbitrary beta tables or copy finished figures.
for index,row in manifest.iterrows():
    recorded = Path(str(row["beta_path"]))
    candidates = [recorded] if recorded.is_absolute() else [manifest_path.parent / recorded, INPUT_MODEL_ROOT / recorded]
    candidates += [manifest_path.parent / "betas_by_replicate" / str(row["day"]) / recorded.name]
    found = next((p for p in candidates if p.is_file() and p.stat().st_size),None)
    if found is None:
        raise FileNotFoundError(f"Missing beta matrix: {recorded}; checked {candidates}")
    manifest.loc[index,"beta_path"] = str(found.resolve())
if manifest.empty:
    raise ValueError("No usable numerical beta matrices in replicate manifest.")
display(manifest)


## S4b — recompute empirical Q-Q coordinates and render per-day panels

In [ ]:

illustrator_out_dir = PANEL_B_DIR

all_saved_files = {}

for day in DAYS:
    p, saved = plot_ipsc_ec_qq(
        qq_points,
        day=day,
        pvalue_type="empirical",
        title=f"{day}: Real target and NTC empirical p-values",
        subtitle="RNA QC, guide count and guide UMI burden",
        output_stem=illustrator_out_dir / f"{day}_QQ_full_covariates",
        formats=("pdf", "svg", "eps", "png"),
    )
    all_saved_files[day] = saved
    display(p)

print("Saved files:")
for day, files in all_saved_files.items():
    print(f"\n{day}")
    for fmt, path in files.items():
        print(f"  {fmt.upper()}: {path}")


## S4c — post-CRT selection from executed source cells 29–31

The prior port stopped at the earlier unfiltered scatterplot. This cell preserves
its raw target-program betas, joins actual target-carrier counts with original cNMF
usage in each replicate, and selects pooled full-day BH q <= 0.10 with >=20 such
cells in BOTH replicates. BH is computed within each day over all real target x
program tests before applying pair/count filters. NTC rows and non-finite effects
are excluded. There is no effect-size trimming and no automatic top-N fallback.

The archived source uses `POOLED_PVALUE_TYPE="skew"` for this selection, with a
recorded empirical fallback when skew files are absent. This differs from S4b's
empirical Q–Q plot. The numerical input audit records which files were actually
used. Saved beta estimates do not need to be recomputed for this correction.
The final source plot style is preserved (point size 1.15, alpha 0.82).


In [ ]:
# Source executed notebook cells 29-31: post-CRT filtering, not a new CRT.
from cc_perturbseq.figure_s04_postcrt import (
    load_or_rebuild_target_counts, read_pooled_results,
    select_effect_pairs, render_selected_effects,
)
PLOT_OUT_DIR = PANEL_C_DIR
unfiltered_effects = build_effect_pair_table(manifest)
unfiltered_path = PANEL_C_DIR / "S4c_unfiltered_real_effect_pairs.csv"
unfiltered_effects.to_csv(unfiltered_path, index=False)

target_counts = load_or_rebuild_target_counts(
    INPUT_MODEL_ROOT, DAYS, PANEL_C_DIR,
    counts_csv=TARGET_COUNTS_CSV,
    mapping_policy=GUIDE_MAPPING_POLICY,
)
pooled, pooled_inputs = read_pooled_results(
    INPUT_MODEL_ROOT, DAYS, preferred=POOLED_PVALUE_TYPE,
)
pooled.to_csv(PANEL_C_DIR / "S4c_pooled_full_day_pvalues_and_qvalues.csv", index=False)
pooled_inputs.to_csv(PANEL_C_DIR / "S4c_pooled_pvalue_input_audit.csv", index=False)
selected, all_pairs_audit, filter_summary = select_effect_pairs(
    unfiltered_effects, pooled, target_counts,
    q_cutoff=POOLED_Q_CUTOFF, min_target_cells=MIN_TARGET_CELLS_PER_REP,
)
all_pairs_audit.to_csv(PANEL_C_DIR / "S4c_filter_audit.csv", index=False)
filter_summary.to_csv(PANEL_C_DIR / "S4c_filter_counts.csv", index=False)
display(pooled_inputs)
display(filter_summary)
# Never silently replace an empty selection with an unfiltered or top-N panel.
effects, summary, combined_plot = render_selected_effects(
    selected, PANEL_C_DIR,
    q_cutoff=POOLED_Q_CUTOFF, min_target_cells=MIN_TARGET_CELLS_PER_REP,
)
display(summary)
display(combined_plot)
selection_tag = (
    f"q_q{str(POOLED_Q_CUTOFF).replace('.', '_')}_top500_mincells{MIN_TARGET_CELLS_PER_REP}"
)
# The source filename includes top500 even though SELECTION_MODE='q'; no fallback runs.
selection_settings_path = PANEL_C_DIR / "S4c_selection_settings.json"
selection_settings_path.write_text(json.dumps({
    "source_executed_notebook_cells": [29, 30, 31],
    "selection_mode": "q", "pooled_q_cutoff": POOLED_Q_CUTOFF,
    "min_target_cells_per_replicate": MIN_TARGET_CELLS_PER_REP,
    "pooled_pvalue_requested": POOLED_PVALUE_TYPE,
    "pooled_pvalue_inputs": pooled_inputs.to_dict("records"),
    "counts_definition": "unique target carriers whose IDs occur in original cNMF usage, each replicate",
    "bh_family": "all finite real target-program tests separately within day, before pair/count filters",
    "no_crt_rerun_required_for_selection": True,
    "historical_counts_are_diagnostic_only": True,
}, indent=2) + "\n")
# Exact outputs from this execution, rather than stale old figures in panel_c/.
selected_output_files = [
    unfiltered_path,
    PANEL_C_DIR / "DIAGNOSTIC_target_cell_counts_by_replicate.csv",
    PANEL_C_DIR / "target_cell_count_provenance.json",
    PANEL_C_DIR / "S4c_pooled_full_day_pvalues_and_qvalues.csv",
    PANEL_C_DIR / "S4c_pooled_pvalue_input_audit.csv",
    PANEL_C_DIR / "S4c_filter_audit.csv",
    PANEL_C_DIR / "S4c_filter_counts.csv",
    selection_settings_path,
    PANEL_C_DIR / f"pooled_selected_replicate_effect_pairs_{selection_tag}.csv",
    PANEL_C_DIR / f"pooled_selected_replicate_effect_correlation_summary_{selection_tag}.csv",
]
stems = [f"pooled_selected_biological_replicate_DE_program_effect_correlation_all_days_{selection_tag}"]
stems += [f"{day}_pooled_selected_biological_replicate_DE_program_effect_correlation_{selection_tag}"
          for day in effects["day"].drop_duplicates()]
selected_output_files += [PANEL_C_DIR / f"{stem}.{suffix}"
                          for stem in stems for suffix in ("pdf", "svg", "png")]
print("Selected S4c outputs:", PANEL_C_DIR)
print("Use the pooled_selected_* figures, not any unfiltered figures from an older run.")


In [ ]:
for panel in ("b", "c"):
    if panel == "b":
        files = [Path(p) for day_outputs in all_saved_files.values() for p in day_outputs.values()]
        files.append(PANEL_B_DIR / "all_days_QQ_points.csv")
    else:
        files = selected_output_files
    if not any(p.suffix == ".svg" and p.is_file() for p in files):
        raise RuntimeError(f"Panel {panel}: no SVG output from this execution")
    write_receipt(
        ctx, panel, status="RESCRIPTED_NUMERICAL_INPUTS_RENDERED",
        inputs={"qq_points": str(qq_csv), "replicate_manifest": str(manifest_path)},
        outputs={p.name: str(p) for p in files if p.is_file()},
        parameters={
            "runtime_source_notebook_required": False,
            "recomputed_crt": RECOMPUTE_CRT,
            "source_style_changed": False,
            "panels": ["b", "c"],
            "s4c_selection": "executed source cells 29-31; pooled q and target-cell support",
            "selection_settings": str(selection_settings_path),
        },
    )
